# Visualise Output Segmentations

This notebook allows you to compare U-Net and ACU-Net outputs in more detail, compared to ground truth segmentations.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 2 levels deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

Update `model_path` to choose the model.

In [ ]:
from data_modules.mnms import MnMsDataModule
import lightning as L
import torch

from arch.unet.acunet import ACUNet
from arch.unet.unet import UNet
from utils.const import SEED
from data_modules.acdc import ACDCDataModule
from utils.utils import setup_device

model_path = "logs/unet_acdc/shape-prior-seed-1970/checkpoints/epoch=74-step=4050.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
# data_module = ACDCDataModule(batch_size=6)
data_module = MnMsDataModule(batch_size=6)

# Load model
model = ACUNet.load_from_checkpoint(model_path)

# Reseed after preprocessing data
L.seed_everything(SEED)

In [ ]:
loader_test = data_module.test_dataloader(shuffle=True)
x, y, _, _ = next(iter(loader_test))
print(x.shape, y.shape)

## View Segmentations

First, let's plot some random instances.

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff

with torch.no_grad():
    model.eval()
    model.to(device)
    x = x.to(device)

    y_hat_logits = model(x)

y_hat_logits = y_hat_logits.cpu()

samples, reconstructions, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(
    y,
    y_hat_logits,
    return_reconstructions=True,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid

from utils.colourmap import GIRIDIS, GREDS

fig, ax = plt.subplots(1, 4, figsize=(24, 36))

# Column 1: Scans

scans = x.cpu().float()
scans = make_grid(scans, nrow=1, padding=2)
scans = np.transpose(scans.numpy(), (1, 2, 0))

ax[0].axis("off")
ax[0].imshow(scans)
ax[0].set_title("Scan", fontsize=36, pad=36)

# Column 2: Ground truth

samples = samples.cpu().float()
samples_grid = make_grid(samples, nrow=1, padding=2, normalize=True)
samples_grid = samples_grid[0]

ax[1].axis("off")
ax[1].imshow(samples_grid, cmap=GIRIDIS)
ax[1].set_title("Ground Truth", fontsize=36, pad=36)

# Column 3: Reconstruction

reconstructions = reconstructions.cpu().float()
reconstructions_grid = make_grid(reconstructions, nrow=1, padding=2, normalize=True)
reconstructions_grid = reconstructions_grid[0]

ax[2].axis("off")
ax[2].imshow(reconstructions_grid, cmap=GIRIDIS)
ax[2].set_title("Reconstruction", fontsize=36, pad=36)

# Column 4: Reconstruction error

reconstruction_pixel_error = reconstruction_pixel_error.cpu().float()
reconstruction_pixel_error_grid = make_grid(reconstruction_pixel_error, nrow=1, padding=2, normalize=True)
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid > 0
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid[0]

ax[3].axis("off")
ax[3].imshow(reconstruction_pixel_error_grid, cmap=GREDS)
ax[3].set_title("Incorrect Pixels", fontsize=36, pad=36)

Now, let's investigate what the **worst** output segmentations look like.

In [ ]:
from utils.eval import compute_dice_score
from utils.utils import discretise

data_module = ACDCDataModule(batch_size=1)

loader_test = data_module.test_dataloader(shuffle=True)

worst_x = []
worst_y = []
worst_y_hat = []

with torch.no_grad():
    for x, y, _, _ in loader_test:
        model.eval()
        model.to(device)
        x = x.to(device)
        y = y.to(device)

        y_hat_logits = model(x)
        y_hat_onehot = discretise(y_hat_logits)
        
        # Compute DSC
        dice_score: torch.Tensor = compute_dice_score(y, y_hat_onehot, device=device)
        
        if dice_score.item() < 0.4:
            worst_x.append(x)
            worst_y.append(y)
            worst_y_hat.append(y_hat_onehot)

print(len(worst_x))

In [ ]:
# Choose 6 samples to view

samples_idx = torch.randperm(len(worst_x))[:6]

worst_x_subset = torch.stack([worst_x[i] for i in samples_idx]).squeeze(1)
worst_y_subset = torch.stack([worst_y[i] for i in samples_idx]).squeeze(1)
worst_y_hat_subset = torch.stack([worst_y_hat[i] for i in samples_idx]).squeeze(1)

worst_x_subset.shape, worst_y_subset.shape, worst_y_hat_subset.shape

In [ ]:
samples, reconstructions, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(
    worst_y_subset,
    worst_y_hat_subset,
    return_reconstructions=True,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid

from utils.colourmap import GIRIDIS, GREDS

fig, ax = plt.subplots(1, 4, figsize=(24, 36))

# Column 1: Scans

scans = worst_x_subset.cpu().float()
scans = make_grid(scans, nrow=1, padding=2)
scans = np.transpose(scans.numpy(), (1, 2, 0))

ax[0].axis("off")
ax[0].imshow(scans)
ax[0].set_title("Scan", fontsize=36, pad=36)

# Column 2: Ground truth

samples = samples.cpu().float()
samples_grid = make_grid(samples, nrow=1, padding=2, normalize=True)
samples_grid = samples_grid[0]

ax[1].axis("off")
ax[1].imshow(samples_grid, cmap=GIRIDIS)
ax[1].set_title("Ground Truth", fontsize=36, pad=36)

# Column 3: Reconstruction

reconstructions = reconstructions.cpu().float()
reconstructions_grid = make_grid(reconstructions, nrow=1, padding=2, normalize=True)
reconstructions_grid = reconstructions_grid[0]

ax[2].axis("off")
ax[2].imshow(reconstructions_grid, cmap=GIRIDIS)
ax[2].set_title("Reconstruction", fontsize=36, pad=36)

# Column 4: Reconstruction error

reconstruction_pixel_error = reconstruction_pixel_error.cpu().float()
reconstruction_pixel_error_grid = make_grid(reconstruction_pixel_error, nrow=1, padding=2, normalize=True)
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid > 0
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid[0]

ax[3].axis("off")
ax[3].imshow(reconstruction_pixel_error_grid, cmap=GREDS)
ax[3].set_title("Incorrect Pixels", fontsize=36, pad=36)